# Heart Disease UCI: Kalp hastalığı var/yok sınıflandırması

Bu başlangıç seviyesi çalışmada doğrulanmış UCI Cleveland verisi kullanılır. `num` hedefi 0 ise hastalık yok, 1–4 ise hastalık var olarak ikili `disease` hedefine dönüştürülür. Pozitif sınıf `1 = hastalık var`dır. Bu notebook eğitim amaçlıdır; klinik tanı aracı değildir.

Hücreleri sırayla çalıştırın. `train.py` ile aynı yordamlar kullanıldığı için veri doğrulaması, öğrenme ve raporlar tekrarlanabilir.

In [ ]:
from IPython.display import display, Markdown
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
import train as proje

pd.set_option('display.max_columns', None)
raw = proje.load_data()
print('UCI Heart Disease — processed Cleveland (1988)')
print('Veri boyutu (satır, sütun):', raw.shape)
display(raw.head())
print('Veri tipleri:')
display(raw.dtypes.rename('veri_tipi').to_frame())
print('Özet istatistikler:')
display(raw.describe().T)

## Sütunların anlamı

Aşağıdaki sözlük UCI Cleveland sürümündeki özellikleri, kategori kodlarını ve birimleri açıklar. `num` kaynak hedefidir; model girdisi değildir. `disease` ikili hedefidir.

In [ ]:
display(pd.DataFrame(proje.DICTIONARY, columns=['Sütun', 'Anlam', 'Birim / kodlar', 'Tür']))

## Temizleme ve sınıf dağılımı

`?` eksik değer belirteci olarak okunur. Kimlik alanları varsa çıkarılır; birebir tekrarlar değerlendirilir. Sayısal kodlu klinik kategoriler kategori olarak işlenir. Median ve en sık değer doldurucuları yalnızca eğitim fold'larında Pipeline tarafından öğrenilecektir.

In [ ]:
df, audit = proje.clean_data(raw)
print('Ham veri / temiz veri boyutları:', audit['raw_shape'], '/', audit['clean_shape'])
print('Temizleme denetimi:')
display(pd.Series(audit, name='değer').to_frame())
print('Özgün num hedefinin sınıf sayıları:')
display(pd.Series(audit['original_target_counts'], name='kayıt_sayısı').rename_axis('num').to_frame())
print('İkili hedef dağılımı:')
display(proje.class_table(df['disease']))
display(Markdown('Sınıfların sayıları ve yüzdeleri yukarıda, gerçek veriden hesaplanmıştır. Sınıf oranları eşit değilse accuracy tek başına yeterli değildir; özellikle hastalık var sınıfının recall değeri de izlenmelidir.'))

## Eğitim/test ayrımı ve model karşılaştırması

Veri %80 eğitim, %20 test olarak `stratify` ve `random_state=42` ile ayrılır. Model seçimi sadece eğitim verisinde beş katlı StratifiedKFold sonuçlarıyla yapılır. Pozitif recall önceliklidir; eşitlikte precision ve sonra F1 ile sıralama yapılır. Test kümesi bu aşamada kullanılmaz.

In [ ]:
X_train, X_test, y_train, y_test = proje.split_data(df)
print(f'Eğitim: {len(X_train)} kayıt (%{len(X_train) / len(df) * 100:.1f}); test: {len(X_test)} kayıt (%{len(X_test) / len(df) * 100:.1f})')
cv_table, selected_name, pipeline = proje.compare_models(X_train, y_train)
print('Eğitim verisinde 5-fold StratifiedKFold karşılaştırması (ortalama ± standart sapma):')
display(cv_table)
print('Seçilen model:', selected_name)
display(Markdown('Model yalnızca eğitim kümesindeki ortalama hastalık-pozitif recall değerine göre seçildi; recall eşitliğinde precision, ardından F1 kullanıldı. Daha yüksek recall daha az hastalık vakasını kaçırmayı hedefler, fakat precision düşebilir ve daha çok yanlış pozitif oluşabilir. Eşik 0.5 olarak sabittir.'))

## Görselleştirmeler

Aşağıdaki grafikler yalnızca eğitim kümesinden üretilir. Her grafiğin hemen ardından, bu veri kümesinde hesaplanan sayılara dayalı kısa yorum gösterilir. Gözlenen ilişkiler nedensellik anlamına gelmez.

In [ ]:
train_df = X_train.assign(disease=y_train)
for kind in ['hedef', 'yas', 'cinsiyet', 'agri', 'nabiz', 'korelasyon']:
    fig, comment = proje.draw_plot(kind, train_df)
    display(fig)
    display(Markdown('**Veriye dayalı yorum:** ' + comment))
    plt.close(fig)

## Son test değerlendirmesi

Seçim tamamlandıktan sonra test kümesi bir kez son değerlendirme için kullanılır. Pozitif sınıf `1 = hastalık var`dır. Confusion matrix satırları gerçek, sütunları tahmin etiketidir.

In [ ]:
metrics, report, matrix, errors = proje.evaluate_final(pipeline, X_test, y_test, X_train)
print('Seçilen model:', selected_name, '| pozitif sınıf: 1 = hastalık var')
display(pd.Series(metrics, name='test_değeri').rename(index={
    'accuracy': 'Accuracy', 'precision': 'Precision', 'recall': 'Recall (hastalık var)', 'f1': 'F1-score'
}).to_frame())
print('Confusion matrix — satırlar gerçek, sütunlar tahmin:')
display(pd.DataFrame(matrix, index=['Gerçek hayır', 'Gerçek evet'], columns=['Tahmin hayır', 'Tahmin evet']))
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(matrix, display_labels=['Hayır', 'Evet']).plot(ax=ax, colorbar=False)
ax.set(xlabel='Tahmin', ylabel='Gerçek etiket', title='Son test: hata matrisi (pozitif = hastalık var)')
fig.tight_layout()
display(fig)
plt.close(fig)
print('Classification report:')
print(report)

### Yanlış tahmin örnekleri ve hata türleri

False negative (yanlış negatif), hastalık gerçekte varken modelin “hastalık yok” demesidir; tıbbi değerlendirmenin gecikmesi riski nedeniyle genellikle daha ciddi sonuçlar doğurabilir. False positive (yanlış pozitif), hastalık yokken “hastalık var” demesidir ve gereksiz kaygı veya ileri incelemeye neden olabilir. Aşağıdaki olası açıklamalar yalnızca gözlemlenebilir özelliklere dayalı ipuçlarıdır; modelin neden yanıldığını kesin olarak kanıtlamaz.

In [ ]:
if errors.empty:
    print('Test kümesinde yanlış tahmin yok.')
else:
    display(errors)
    print('Yanlış tahmin sayısı:', int((pipeline.predict(X_test) != y_test).sum()), '| tabloda en fazla 5 örnek gösterilir.')
print('Bu proje eğitim amaçlıdır; model, olasılıkları ve hata örnekleri klinik tanı veya doğrulanmış tıbbi risk tahmini değildir.')
proje.save_results(pipeline, cv_table, selected_name, metrics, report, matrix, errors, audit, X_train, X_test, y_train, y_test)
print('Pipeline ve raporlar artifacts/ ile reports/ klasörlerine kaydedildi.')